# TP PAC-Bayes -- Session 1: Foundations: kl inversion, first forest certificates, forest size

Solution notebook for Session 1 (three hours, Parts 1.1 (setup) to 1.4 of the sheet `tp_pacbayes.pdf`). It reproduces the numbers, tables (written to `results/`) and figures (written to `figures/`) of the solution sheet `tp_pacbayes_solutions.pdf`.

It needs `pbtools.py`, `prepdata.py` and the folder `datasets/` of the course, either in this folder or in its parent folder. Each session notebook is self-contained: the helper functions it needs are redefined at the top.

In [1]:
import os, sys, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
for p in [HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data"), os.path.join(HERE, "..", "..", "common"), os.path.join(HERE, "data"), os.path.join(HERE, "..", "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)
from pbtools import (kl_bin, kl_inv_upper, kl_inv_lower, kl_div, bound_seeger, bound_mcallester,
                     BaggedTrees, mv_statistics, fo_bound, tnd_bound, cbound_pb, softmax,
                     optimize_fo, optimize_tnd)
from prepdata import data_recovery

DATADIR = next((d for d in [os.path.join(HERE, "data"), HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data")] if os.path.isdir(os.path.join(d, "datasets"))), HERE)
RES = os.path.join(HERE, "results"); FIG = os.path.join(HERE, "figures")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)
DELTA = 0.05
N_SPLITS = 5
BALANCED = ["australian", "balance", "bupa", "german", "heart", "iono", "pima", "sonar", "wdbc", "spambase"]
IMBALANCED = ["abalone8", "pageblocks", "satimage", "segmentation", "yeast3"]
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})


def load(name):
    """Load a dataset of the course with labels in {-1,+1} and standardized features."""
    cwd = os.getcwd(); os.chdir(DATADIR)
    try:
        X, y = data_recovery(name)
    finally:
        os.chdir(cwd)
    X = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    return X, np.where(np.asarray(y) > 0, 1, -1)


def eff(rho):
    """Effective number of voters 1/sum rho^2."""
    return 1.0 / np.sum(rho ** 2)


def save_tab(name, rows):
    open(os.path.join(RES, name), "w").write("\n".join(rows) + "\n")
    print("  -> results/" + name)


def ms(a, d=3):
    a = np.asarray(a, dtype=float)
    return f"{a.mean():.{d}f} ({a.std():.{d}f})"

## Part 1.2 -- Warm-up: inverting the kl and certifying one classifier

In [2]:
def kl_inv_bisection(q, eps, tol=1e-9):
    """Our own implementation of the upper kl inversion (Algorithm 1 of the notes)."""
    a, b = q, 1.0
    while b - a > tol:
        c = (a + b) / 2
        if kl_bin(q, c) > eps:
            b = c
        else:
            a = c
    return b


def part1():
    print("Part 1")
    rows = []
    for m in [100, 1000]:
        eps = np.log(1 / DELTA) / m
        for q in [0.0, 0.01, 0.05, 0.1, 0.3, 0.5]:
            k = kl_inv_bisection(q, eps)
            assert abs(k - kl_inv_upper(q, eps)) < 1e-6
            rows.append(f"{m} & {q:.2f} & {q + np.sqrt(eps / 2):.3f} & {k:.3f} & {q + np.sqrt(2 * q * eps) + 2 * eps:.3f} \\\\")
    save_tab("p1_widths.tex", rows)
    # certify one tree with an independent test sample (test-set bound)
    X, y = load("australian")
    X1, X2, y1, y2 = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)
    tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X1, y1)
    err = np.mean(tree.predict(X2) != y2); n = len(y2)
    hoef = err + np.sqrt(np.log(1 / DELTA) / (2 * n)); kl = kl_inv_upper(err, np.log(1 / DELTA) / n)
    # naive and corrected bounds after choosing the depth on the same half
    errs = {d: np.mean(DecisionTreeClassifier(max_depth=d, random_state=0).fit(X1, y1).predict(X2) != y2) for d in range(1, 11)}
    dbest = min(errs, key=errs.get)
    naive = kl_inv_upper(errs[dbest], np.log(1 / DELTA) / n)
    union = kl_inv_upper(errs[dbest], np.log(10 / DELTA) / n)
    save_tab("p1_tree.tex", [f"{n} & {err:.3f} & {hoef:.3f} & {kl:.3f} & {dbest} & {errs[dbest]:.3f} & {naive:.3f} & {union:.3f}"])

In [3]:
t0 = time.time(); part1(); print(f'{time.time() - t0:.0f} s')

Part 1
  -> results/p1_widths.tex
  -> results/p1_tree.tex
0 s


## Parts 1.3 and 2.1 -- Certificates of a standard random forest (and learned weights, reused in Session 2)

In [4]:
def forest_stats(X, y, seed, n_trees=100, frac=0.5, max_features="sqrt", max_depth=None):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    bt = BaggedTrees(n_trees, bootstrap_frac=frac, max_features=max_features, max_depth=max_depth,
                     random_state=seed).fit(Xtr, ytr)
    L, n1 = bt.oob_losses(Xtr, ytr); Lt, n2 = bt.oob_tandem(Xtr, ytr); D, n3 = bt.oob_disagreement(Xtr)
    return dict(bt=bt, Xtr=Xtr, Xte=Xte, ytr=ytr, yte=yte, L=L, n1=n1, Lt=Lt, n2=n2, D=D, n3=n3,
                Vte=bt.predict_all(Xte), pi=np.ones(n_trees) / n_trees)


def certificates(F, rho):
    return (fo_bound(rho, F["pi"], F["L"], F["n1"], DELTA), tnd_bound(rho, F["pi"], F["Lt"], F["n2"], DELTA),
            cbound_pb(rho, F["pi"], F["L"], F["n1"], F["D"], F["n3"], DELTA))


CACHE = {}


def part2_3():
    print("Part 2 and 3")
    rows2, rows3, summary = [], [], {k: [] for k in ["u", "fo", "tnd"]}
    for name in BALANCED + IMBALANCED:
        X, y = load(name)
        R = {k: [] for k in ["mv", "gibbs", "dis", "fo", "tnd", "cb", "rf", "n1", "n2",
                             "mv_fo", "cert_fo", "eff_fo", "mv_tnd", "cert_tnd", "eff_tnd", "best_u"]}
        for seed in range(N_SPLITS):
            F = forest_stats(X, y, seed)
            CACHE[(name, seed)] = F
            st = mv_statistics(F["Vte"], F["yte"], F["pi"])
            fo, tnd, cb = certificates(F, F["pi"])
            rf = RandomForestClassifier(100, random_state=seed).fit(F["Xtr"], F["ytr"])
            for k, v in [("mv", st["mv"]), ("gibbs", st["gibbs"]), ("dis", st["dis"]), ("fo", fo), ("tnd", tnd),
                         ("cb", cb), ("rf", np.mean(rf.predict(F["Xte"]) != F["yte"])), ("n1", F["n1"]), ("n2", F["n2"])]:
                R[k].append(v)
            # Part 3: learning the weights
            r_fo = optimize_fo(F["pi"], F["L"], F["n1"], DELTA)
            r_tnd = optimize_tnd(F["pi"], F["Lt"], F["n2"], DELTA)
            R["mv_fo"].append(mv_statistics(F["Vte"], F["yte"], r_fo)["mv"])
            R["cert_fo"].append(fo_bound(r_fo, F["pi"], F["L"], F["n1"], DELTA)); R["eff_fo"].append(eff(r_fo))
            R["mv_tnd"].append(mv_statistics(F["Vte"], F["yte"], r_tnd)["mv"])
            R["cert_tnd"].append(tnd_bound(r_tnd, F["pi"], F["Lt"], F["n2"], DELTA)); R["eff_tnd"].append(eff(r_tnd))
            # best of the three certificates of the uniform vote, with a union bound (delta/3)
            R["best_u"].append(min(fo_bound(F["pi"], F["pi"], F["L"], F["n1"], DELTA / 3),
                                   tnd_bound(F["pi"], F["pi"], F["Lt"], F["n2"], DELTA / 3),
                                   cbound_pb(F["pi"], F["pi"], F["L"], F["n1"], F["D"], F["n3"], DELTA / 3)))
        M = {k: np.mean(v) for k, v in R.items()}
        rows2.append(f"{name} & {M['rf']:.3f} & {M['mv']:.3f} & {M['gibbs']:.3f} & {M['dis']:.3f} & {int(M['n1'])} & {int(M['n2'])} & "
                     f"{M['fo']:.3f} & {M['tnd']:.3f} & {M['cb']:.3f} \\\\")
        rows3.append(f"{name} & {ms(R['mv'])} & {ms(R['best_u'])} & {ms(R['mv_fo'])} & {ms(R['cert_fo'])} & {np.mean(R['eff_fo']):.1f} & "
                     f"{ms(R['mv_tnd'])} & {ms(R['cert_tnd'])} & {np.mean(R['eff_tnd']):.1f} \\\\")
        summary["u"].append((M["mv"], M["best_u"])); summary["fo"].append((M["mv_fo"], M["cert_fo"])); summary["tnd"].append((M["mv_tnd"], M["cert_tnd"]))
        print(f"  {name}: mv {M['mv']:.3f} FO {M['fo']:.3f} TND {M['tnd']:.3f} CB {M['cb']:.3f} | FO-opt {M['mv_fo']:.3f}/{M['cert_fo']:.3f} TND-opt {M['mv_tnd']:.3f}/{M['cert_tnd']:.3f}")
    s = {k: np.mean(v, axis=0) for k, v in summary.items()}
    rows3.append(r"\midrule")
    rows3.append(f"mean & {s['u'][0]:.3f} & {s['u'][1]:.3f} & {s['fo'][0]:.3f} & {s['fo'][1]:.3f} & & {s['tnd'][0]:.3f} & {s['tnd'][1]:.3f} & \\\\")
    save_tab("p2_forest.tex", rows2); save_tab("p3_weights.tex", rows3)

In [5]:
t0 = time.time(); part2_3(); print(f'{time.time() - t0:.0f} s')

Part 2 and 3


  australian: mv 0.141 FO 0.649 TND 0.819 CB 0.839 | FO-opt 0.170/0.509 TND-opt 0.138/0.772


  balance: mv 0.097 FO 0.523 TND 0.664 CB 0.731 | FO-opt 0.137/0.442 TND-opt 0.101/0.633


  bupa: mv 0.254 FO 1.000 TND 1.000 CB 1.000 | FO-opt 0.367/0.985 TND-opt 0.283/1.000


  german: mv 0.242 FO 0.864 TND 1.000 CB 0.968 | FO-opt 0.297/0.761 TND-opt 0.248/0.998


...

  abalone8: mv 0.141 FO 0.502 TND 0.591 CB 0.639 | FO-opt 0.178/0.472 TND-opt 0.143/0.572


  pageblocks: mv 0.021 FO 0.124 TND 0.150 CB 0.190 | FO-opt 0.034/0.105 TND-opt 0.022/0.140


  satimage: mv 0.062 FO 0.248 TND 0.271 CB 0.329 | FO-opt 0.094/0.227 TND-opt 0.061/0.258


  segmentation: mv 0.027 FO 0.210 TND 0.221 CB 0.310 | FO-opt 0.049/0.152 TND-opt 0.025/0.199


  yeast3: mv 0.046 FO 0.274 TND 0.352 CB 0.424 | FO-opt 0.053/0.220 TND-opt 0.048/0.325
  -> results/p2_forest.tex
  -> results/p3_weights.tex
143 s


## Part 1.4 -- Number of trees and size of the bootstrap samples

In [6]:
def partA():
    print("Part A (trees / bootstrap)")
    rows = []
    X, y = load("spambase")
    for frac in [0.3, 0.5, 0.8, 1.0]:
        for n in [10, 50, 100, 200]:
            R = []
            for seed in range(3):
                F = forest_stats(X, y, seed, n_trees=n, frac=frac)
                st = mv_statistics(F["Vte"], F["yte"], F["pi"])
                R.append((st["mv"], F["n1"], F["n2"], fo_bound(F["pi"], F["pi"], F["L"], F["n1"], DELTA),
                          tnd_bound(F["pi"], F["pi"], F["Lt"], F["n2"], DELTA)))
            R = np.mean(R, 0)
            rows.append(f"{frac} & {n} & {R[0]:.3f} & {int(R[1])} & {int(R[2])} & {R[3]:.3f} & {R[4]:.3f} \\\\")
    save_tab("pA_trees.tex", rows)

In [7]:
t0 = time.time(); partA(); print(f'{time.time() - t0:.0f} s')

Part A (trees / bootstrap)


  -> results/pA_trees.tex
44 s
